# 01 · Claude Agent SDK 入門

> 目標：40 分鐘內，從「零」到「一個有自訂工具、而且看得見它在想什麼的 agent」。
> 下一本 `02_compose_architectures.ipynb` 才開始組 CRAG / Self-RAG / Adaptive-RAG。

## 先講清楚三個東西的差別

學生最容易在這裡搞混，先用一張表解決：

| | 是什麼 | 誰跑 agent loop | 能用什麼工具 |
|---|---|---|---|
| **Claude API**（`anthropic`） | 原始 Messages API | **你自己寫 while 迴圈** | 你定義的 |
| **Tool Runner**（`client.beta.messages.tool_runner`） | API SDK 的小幫手 | SDK | 你定義的 |
| **Claude Agent SDK**（`claude-agent-sdk`） | **Claude Code 包成函式庫** | SDK | 內建 Read/Write/Bash/WebSearch + 你的 MCP 工具 |

這本用的是**第三個**。它底層會 spawn `claude` CLI —— 這件事很重要，因為：

> **CLI 吃什麼憑證，你的 agent 就吃什麼憑證。** 所以可以直接用 Claude 訂閱的 OAuth，不用 API key、不產生 API 帳單。

## Step 0 · 認證：四選一

| 接法 | 怎麼設 | 費用 |
|---|---|---|
| 本機已登入 Claude Code | 什麼都不用做 | 訂閱額度 |
| OAuth token | `claude setup-token` → `CLAUDE_CODE_OAUTH_TOKEN` | 訂閱額度，**無 API 帳單** |
| Anthropic 相容端點 | `ANTHROPIC_BASE_URL` + `ANTHROPIC_AUTH_TOKEN` + `ANTHROPIC_MODEL` | 該 provider 計費 |
| Anthropic 官方 API | `ANTHROPIC_API_KEY` | 官方計費 |

第三種可以接 DeepSeek / Kimi / GLM / OpenRouter，或自架 LiteLLM proxy 再轉去任何模型。
**下面的程式碼在四種情況下完全相同**，差別只在環境變數。

In [1]:
# ── 這格在做什麼：把環境準備好，讓後面每一格都 import 得到專案的程式碼 ──
import os, sys, json
from pathlib import Path

# 這個 notebook 放在 notebooks/ 子資料夾，但 retrieval.py、modules.py 都在上一層（專案根目錄）。
# Python 預設只在「目前工作目錄」找模組，所以要先把工作目錄切到專案根，
# 再把它加進 sys.path（Python 找模組的路徑清單），之後 import 才找得到。
# 先判斷 cwd().name == "notebooks"，是為了讓這格重跑幾次都不會一路往上跳。
if Path.cwd().name == "notebooks":
    os.chdir("..")                       # 往上一層
sys.path.insert(0, str(Path.cwd()))      # 插在最前面 = 優先從專案根找模組

# .env 是放認證資訊的檔案（例如 ANTHROPIC_API_KEY=...）。
# load_dotenv() 把裡面每一行讀進環境變數，之後 os.getenv() 就拿得到。
from dotenv import load_dotenv
load_dotenv()

import claude_agent_sdk
print("claude-agent-sdk", claude_agent_sdk.__version__)   # 印版本，確認套件裝對了

# 對照 Step 0 的表格，看你設的是四種認證裡的哪一種。
# 語法注意：這是 Python 的 for...else。這裡的 else 不是「否則」，
# 是「迴圈跑完都沒有 break 才執行」，意思就是「四個都沒設」。
for key in ["CLAUDE_CODE_OAUTH_TOKEN", "ANTHROPIC_BASE_URL", "ANTHROPIC_AUTH_TOKEN", "ANTHROPIC_API_KEY"]:
    if os.getenv(key):
        print(f"偵測到 {key}")
        break
else:
    print("沒有設任何認證環境變數 —— 如果這台機器的 claude CLI 已登入，SDK 會直接沿用")

claude-agent-sdk 0.2.152
沒有設任何認證環境變數 —— 如果這台機器的 claude CLI 已登入，SDK 會直接沿用


## Step 1 · 最小可跑：一問一答

`query()` 是最簡單的入口：丟一個 prompt，串流回來一堆 message。

先不要給任何工具（`tools=[]` 把內建的 Read / Write / Bash 全關掉），
確認管線通了再往下。

In [2]:
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock

# query() 是 SDK 最簡單的入口：給一個 prompt，它在背景啟動 claude CLI，
# 然後把模型回來的東西「一則一則」串流給你 —— 所以用 async for 逐則接，不是一次拿到整包。
#
# 為什麼可以直接寫 async for？因為 Jupyter 本身就跑在 asyncio 事件迴圈裡，
# 格子裡允許直接 await。如果把這段搬去一般的 .py 檔，
# 要包進 async def main() 再用 asyncio.run(main()) 執行。
async for message in query(
    prompt="用一句話說明 RAG 是什麼，不要超過 30 個字。",
    options=ClaudeAgentOptions(
        tools=[],            # 關掉所有內建工具（Read / Write / Bash…）。第一次先確認管線通就好
        setting_sources=[],  # 不要載入這個 repo 的 CLAUDE.md、hooks 等專案設定
        max_turns=1,         # 最多讓模型回一輪。沒有工具本來也只需要一輪
    ),
):
    # 串流回來的 message 有好幾種：AssistantMessage（模型說的話）、
    # SystemMessage（系統訊息）、ResultMessage（結束時的統計）…
    # 我們只關心模型說了什麼，所以用 isinstance 篩出 AssistantMessage。
    if isinstance(message, AssistantMessage):
        # 一則 AssistantMessage 的 content 是「區塊」清單，區塊可能是
        # TextBlock（文字）、ToolUseBlock（要呼叫工具）、ThinkingBlock（思考）…
        # 這格只印文字區塊。
        for block in message.content:
            if isinstance(block, TextBlock):
                print(block.text)

RAG 是讓模型先檢索外部資料，再依據檢索結果生成答案的技術。


### ⚠️ 兩個選項現在就要養成習慣寫

| 選項 | 為什麼 |
|---|---|
| `tools=[]` | 關掉**所有**內建工具。不寫的話 agent 有 Bash / Write / Read —— 一個只該做檢索的 agent 不需要能刪你的檔案。 |
| `setting_sources=[]` | 不要載入這個 repo 自己的 `CLAUDE.md` 和 hooks。不然你的教學 agent 會繼承一堆無關的專案規則。 |

> **踩過的坑**：只設 `allowed_tools` 是**擋不住**內建工具的。
> `allowed_tools` 是「不用問就能用」的白名單，不是「只能用這些」。要真的關掉必須用 `tools=[]`。

## Step 2 · 第一個自訂工具

`@tool` 裝飾器 + `create_sdk_mcp_server`，把一個普通的 Python 函式變成 agent 能呼叫的工具。

「in-process MCP server」聽起來很嚇人，其實就是**跑在你同一個 Python 程序裡的工具集**——
不用另外開 server、不用寫設定檔。

### 為什麼拿擲骰子當例子？

因為它是最小的「模型自己做不到、一定得呼叫工具」的例子：模型沒有亂數產生器，
要嘛呼叫工具，要嘛自己編一個數字 —— 而「編數字」這件事，Step 3 用 hook 一抓就抓到。

先講清楚名詞：**骰子不一定是六面的**。桌遊（例如 D&D）常用 4 / 8 / 12 / 20 面的骰子，
「20 面骰」擲出來是 1～20 的整數。所以工具多開一個 `sides`（面數）參數，
沒給就當一般的六面骰。

等一下的 prompt 會故意說「三顆 20 面骰」，考模型兩件事：

1. 把「20 面」翻成參數 `sides=20` —— 它只看得到工具描述，所以描述要寫清楚
2. 把「三顆」翻成**呼叫三次**，而不是呼叫一次再自己乘三、或乾脆編一個總和

In [3]:
from claude_agent_sdk import tool, create_sdk_mcp_server

# @tool 是一個「裝飾器」：貼在函式上方，等於幫這個函式掛上三張說明牌，
# 讓模型知道有這個工具、它做什麼、要給什麼參數。三個參數依序是：
@tool(
    "roll_dice",                                   # ① 工具名稱：模型呼叫時用這個名字
    "擲一顆骰子。sides 是面數，預設 6。",             # ② 描述：模型「只看得到這一句」，靠它決定何時用、怎麼填參數
    {"type": "object",                             # ③ 參數規格（JSON Schema 格式）：
     "properties": {"sides": {"type": "integer"}}, #    有一個叫 sides 的整數參數
     "required": []},                              #    required 是空的 → sides 可以不給
)
async def roll_dice(args):
    # args 是模型填的參數，型別是 dict，例如 {"sides": 20}。
    # 模型沒填 sides 時 args 會是 {}，所以用 .get("sides", 6) 給預設值 6。
    import random
    n = random.randint(1, args.get("sides", 6))    # 1 ～ sides 之間隨機取一個整數
    # 回傳格式是 MCP 規定的，長相固定：content 是清單，裡面放 text 區塊。
    # 文字內容自己決定，這裡用 JSON 字串 {"result": 17}，模型讀起來最不會誤會。
    return {"content": [{"type": "text", "text": json.dumps({"result": n})}]}

# 把工具打包成一個「MCP server」。三個參數：server 名稱、版本、工具清單。
# 「server」聽起來要另外開程式，其實沒有 —— 它就跑在這個 notebook 的 Python 程序裡。
# server 名稱 "demo" 待會會變成工具全名的一部分（見下一格）。
dice_server = create_sdk_mcp_server("demo", "1.0.0", [roll_dice])
print("工具做好了")

工具做好了


### 工具名稱會被加上前綴

註冊之後，模型看到的名字是 `mcp__<server 名>__<工具名>`，
也就是 `mcp__demo__roll_dice`。`allowed_tools` 要寫**完整名稱**。

In [4]:
# 把工具掛給 agent。每個選項都有它的理由：
options = ClaudeAgentOptions(
    tools=[],                                      # 內建工具（Read/Write/Bash…）全關，只留我們自己的
    mcp_servers={"demo": dice_server},             # 掛上剛做好的 server。key 要跟 server 名稱一致
    allowed_tools=["mcp__demo__roll_dice"],        # 「不用問就能用」的白名單。要寫 mcp__<server>__<工具> 全名
    strict_mcp_config=True,                        # 只認這裡給的 mcp_servers，忽略專案裡的 .mcp.json
    setting_sources=[],                            # 不載入這個 repo 的 CLAUDE.md / hooks
    system_prompt="你是一個擲骰助手。使用者要你擲骰就呼叫工具，不要自己編數字。用繁體中文回答，不要說開場白。",
    max_turns=5,                                   # 一次工具呼叫來回算一輪。擲三次至少 3 輪 + 1 輪作答，給 5 剛好
)

# 「三顆 20 面骰」= 20 面骰擲三次、各得 1～20、加總。
# 模型要自己想到 sides=20，而且要呼叫三次工具。
# 這格只看得到最後答案 —— 到底有沒有真的呼叫三次，Step 3 用 hook 才看得見。
async for message in query(prompt="幫我擲三顆 20 面骰，然後告訴我總和。", options=options):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print(block.text)

三顆 20 面骰結果：**11、8、8**

總和：**27**


## Step 3 · 用 hook 看見 agent 在做什麼

上面那格只看得到最後結果。**agent 中間呼叫了幾次工具、參數是什麼，全都看不到。**

`PreToolUse` / `PostToolUse` hook 可以攔截每一次工具呼叫。這是把黑盒子打開的關鍵，
也是下一本 notebook 用來比較不同 RAG 架構的基礎 —— 沒有它就沒辦法說明「CRAG 比 Naive 多做了什麼」。

hook callback 的簽名固定是 `(input, tool_use_id, context)`，回傳 `{}` 代表放行。

In [5]:
from claude_agent_sdk import HookMatcher

# hook = 「工具被呼叫前 / 後，SDK 先來敲一下你寫的函式」。
# 用一個 list 把每次敲門的內容記下來，跑完再印 —— 這就是「軌跡（trace）」。
trace = []

# PreToolUse：工具「即將」被呼叫時觸發。
# data 裡有 tool_name（工具全名）和 tool_input（模型填的參數）。
# (data, tool_use_id, context) 這三個參數是 SDK 規定的簽名，用不到的也要收。
async def on_pre_tool(data, tool_use_id, context):
    # 工具全名是 mcp__demo__roll_dice，太長。rsplit("__", 1) 從右邊切一刀，[-1] 拿最後一段 → roll_dice
    trace.append(("呼叫", data["tool_name"].rsplit("__", 1)[-1], data["tool_input"]))
    return {}   # 回傳空 dict = 放行。要擋下這次呼叫的話，回傳的 dict 要帶拒絕的決定（格式見 SDK 文件）

# PostToolUse：工具「執行完」觸發。data["tool_response"] 是工具回傳的 content 清單。
async def on_post_tool(data, tool_use_id, context):
    blocks = data.get("tool_response")
    # 正常情況 blocks 長 [{"type": "text", "text": "..."}]，取第一塊的 text。
    # 萬一格式不是預期的（例如錯誤訊息），就整包轉字串，至少不會炸掉。
    text = blocks[0]["text"] if isinstance(blocks, list) and blocks else str(blocks)
    trace.append(("回傳", data["tool_name"].rsplit("__", 1)[-1], text[:80]))   # 只留前 80 字
    return {}

# 跟上一格幾乎一樣，只多了 hooks 這個選項。
options_with_hooks = ClaudeAgentOptions(
    tools=[],
    mcp_servers={"demo": dice_server},
    allowed_tools=["mcp__demo__roll_dice"],
    strict_mcp_config=True,
    setting_sources=[],
    system_prompt="你是一個擲骰助手。使用者要你擲骰就呼叫工具，不要自己編數字。用繁體中文回答，不要說開場白。",
    max_turns=8,
    # hooks 的結構：{事件名稱: [HookMatcher, ...]}。
    # HookMatcher 可以用 matcher= 指定只攔某些工具；沒指定就是全部工具都攔。
    hooks={
        "PreToolUse":  [HookMatcher(hooks=[on_pre_tool])],
        "PostToolUse": [HookMatcher(hooks=[on_post_tool])],
    },
)

async for message in query(prompt="擲三顆 20 面骰，告訴我總和。", options=options_with_hooks):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print("模型：", block.text)

# 跑完後把軌跡印出來。預期看到「呼叫 roll_dice {'sides': 20}」三次，每次後面跟一個「回傳」。
# {kind:4} 是 f-string 的對齊語法：這一欄固定佔 4 格寬，輸出才排得整齊。
print("\n── 軌跡 ──")
for kind, name, payload in trace:
    print(f"{kind:4} {name:12} {payload}")

模型： 三顆 d20 結果：7、1、17

總和：**25**



── 軌跡 ──
呼叫   roll_dice    {'sides': 20}
回傳   roll_dice    {"result": 7}
呼叫   roll_dice    {'sides': 20}
回傳   roll_dice    {"result": 1}
呼叫   roll_dice    {'sides': 20}
回傳   roll_dice    {"result": 17}


### 看到什麼了？

軌跡會顯示它**真的擲了三次**，而不是擲一次然後自己乘以三、或乾脆編一個數字。

> **教學金句**：「沒有 hook 的 agent 是一個你只能相信的黑盒子。有 hook 的 agent 是一個你可以驗證的系統。」

這件事在 RAG 裡更重要：你怎麼知道模型的答案是「檢索到的」還是「它本來就知道的」？
看它有沒有真的去查、查了什麼。

## Step 4 · `ClaudeSDKClient`：需要多輪對話時用

`query()` 是一次性的。要在同一個 session 裡連續問、或中途插話，就用 `ClaudeSDKClient`：

```python
async with ClaudeSDKClient(options=options) as client:
    await client.query("第一個問題")
    async for msg in client.receive_response(): ...

    await client.query("接著問")          # 同一個 session，有前面的上下文
    async for msg in client.receive_response(): ...
```

下一本 notebook 用的是這個，因為要在一次執行裡跑完整個 RAG 流程並收集軌跡。

## 你現在會的東西

| 會了 | 用在哪 |
|---|---|
| `query()` / `ClaudeSDKClient` | 跑 agent |
| `@tool` + `create_sdk_mcp_server` | 把 Python 函式變成 agent 能用的工具 |
| `ClaudeAgentOptions(tools=[], setting_sources=[])` | 把 agent 關進安全的小房間 |
| `HookMatcher` + PreToolUse/PostToolUse | 看見 agent 每一步在做什麼 |

**這四樣就足夠組出任何 RAG 架構了。** 下一本開始組。

---

## 卡點對照表

| 卡點 | 原因 | 處理 |
|---|---|---|
| `ModuleNotFoundError: retrieval` | notebook 的工作目錄在 `notebooks/` | 跑第一格（會自動 `chdir("..")`） |
| agent 完全不呼叫工具 | 工具描述寫太模糊，或 system prompt 沒說要用 | 把 description 寫具體，prompt 明講「要用工具不要自己編」 |
| `allowed_tools` 寫了工具名還是被擋 | 忘記 `mcp__<server>__` 前綴 | 用完整名稱 |
| 設了 `tools=[]` 但 MCP 工具也不見了 | 誤會 —— `tools` 只管內建工具 | MCP 工具走 `mcp_servers`，兩者獨立 |
| 每次跑都很慢 | 每次 `query()` 都重新 spawn CLI | 多輪就用 `ClaudeSDKClient` 開一次連線 |

---

_下一本：`02_compose_architectures.ipynb` —— 用這四樣東西組出 CRAG、Self-RAG、Adaptive-RAG_